# Лучшие параметры embedding по колонкам

Ноутбук читает полный результат перебора `uzal_embedding_results_all_columns.csv` и оставляет по одной лучшей строке на каждую колонку: минимальный `uzal_cost`, соответствующие `tau` и `dimension`.

In [1]:
from pathlib import Path

import pandas as pd
import plotly.express as px

In [8]:
import subprocess

process = subprocess.Popen(
    ["julia", "--project=.", "main.jl"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end="")

return_code = process.wait()

if return_code != 0:
    raise subprocess.CalledProcessError(return_code, process.args)

Numeric columns: 117
[1/117] Lab1_G1_N1
[2/117] Lab1_G1_N2
[3/117] Lab1_G1_N3
[4/117] Lab1_G1_P2
[5/117] Lab1_G1_T4ср
[6/117] Lab1_G1_T1
[7/117] Lab1_G1_T607
[8/117] Lab1_G1_T600
[9/117] Lab1_G1_T638
[10/117] Lab1_G1_T606
[11/117] Lab1_G1_T1002
[12/117] Lab1_G1_T1003
[13/117] Lab1_G2_Fc2
[14/117] Lab1_G2_F1
[15/117] Lab1_G2_Fc3
[16/117] Lab1_G2_2F1
[17/117] Lab1_G2_3F1
[18/117] Lab1_G2_Fтк2
[19/117] Lab1_G2_Fнш
[20/117] Lab1_G2_Fма
[21/117] Lab1_G2_Fмн
[22/117] Lab1_G2_Fств
[23/117] Lab1_G2_Fc4
[24/117] Lab1_G2_Fцс
[25/117] Lab1_G2_F2
[26/117] Lab1_G2_Fкпа
[27/117] Lab1_G2_2F2
[28/117] Lab1_G2_3F2
[29/117] Lab1_G2_Fтк4
[30/117] Lab1_G2_3_77F2
[31/117] Lab1_G2_VoГГ
[32/117] Lab1_G2_Fc9
[33/117] Lab1_G2_Fс8
[34/117] Lab1_G2_F3
[35/117] Lab1_G2_2F3
[36/117] Lab1_G2_3F3
[37/117] Lab1_G2_Fтк9
[38/117] Lab1_G2_Fтк8
[39/117] Lab1_G2_Fн9
[40/117] Lab1_G2_Fв9
[41/117] Lab1_G2_VoСТ
[42/117] Lab1_G3_N3
[43/117] Lab1_G3_Lm
[44/117] Lab1_G3_dPf1
[45/117] Lab1_G3_Pm
[46/117] Lab1_G3_T638
[47/117] La

In [9]:
RESULTS_PATH = Path("uzal_embedding_results_all_columns.csv")
SKIPPED_PATH = Path("uzal_embedding_skipped_columns.csv")
BEST_OUTPUT_PATH = Path("uzal_best_choices_by_column.csv")

In [10]:
results = pd.read_csv(RESULTS_PATH)
results.head(), results.shape

(       column  tau  dimension  uzal_cost
 0  Lab1_G1_N1   98          4   1.026301
 1  Lab1_G1_N1   93          5   1.026900
 2  Lab1_G1_N1   74          5   1.027976
 3  Lab1_G1_N1   72          6   1.035373
 4  Lab1_G1_N1   98          6   1.036983,
 (88000, 4))

In [11]:
required_columns = {"column", "tau", "dimension", "uzal_cost"}
missing_columns = required_columns - set(results.columns)
if missing_columns:
    raise ValueError(f"Missing columns in {RESULTS_PATH}: {sorted(missing_columns)}")

all_result_columns = set(results["column"].dropna())
finite_results = results.dropna(subset=["column", "tau", "dimension", "uzal_cost"]).copy()
finite_results["tau"] = finite_results["tau"].astype(int)
finite_results["dimension"] = finite_results["dimension"].astype(int)

print(f"Rows in full results: {len(results)}")
print(f"Columns in full results: {len(all_result_columns)}")
print(f"Rows with finite uzal_cost: {len(finite_results)}")
print(f"Columns with finite uzal_cost: {finite_results['column'].nunique()}")

Rows in full results: 88000
Columns in full results: 88
Rows with finite uzal_cost: 61280
Columns with finite uzal_cost: 74


In [12]:
best_idx = finite_results.groupby("column")["uzal_cost"].idxmin()

best_choices = (
    finite_results.loc[best_idx, ["column", "tau", "dimension", "uzal_cost"]]
    .sort_values("uzal_cost")
    .reset_index(drop=True)
)

pd.set_option("display.max_rows", None)

display(best_choices)

,column,tau,dimension,uzal_cost
0,Lab1_G3_Lm,30,3,-3.305123
1,Lab1_Rc,80,11,-2.885113
2,Lab1_Hpol,69,5,-2.501856
3,Lab1_he,23,8,-2.348359
4,Lab1_dev,9,4,-2.216724
5,Lab1_G3_T638,13,3,-1.933533
6,Lab1_G3_T600,20,3,-1.916388
7,Lab1_PdoNag,79,5,-1.823454
8,Lab1_Pm_sm_N,96,10,-1.807554
9,Lab1_dPmg,88,9,-1.807238


In [13]:
from pathlib import Path

paths = [
    Path("uzal_columns_without_finite_cost.csv"),
    Path("uzal_embedding_results_all_columns.csv"),
    Path("uzal_embedding_results.csv"),
    Path("uzal_embedding_skipped_columns.csv"),
    Path("uzal_best_choices_by_column.csv")
]
  
for path in paths:
    if path.exists():
        path.unlink()

best_choices.to_csv("uzal_cost_final_result.csv")